In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes -q

!pip install fastapi uvicorn pinggy -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 20.3 MB/s eta 0:00:00
ERROR: Operation cancelled by user
^C


Tải Mô hình Llama-3 và Khởi chạy FastAPI ngầm

In [ ]:
import threading
import uvicorn
import re
from fastapi import FastAPI
from unsloth import FastLanguageModel

app = FastAPI()

MODEL_ID = "tazuneru/llama-3-8b-banking-intent"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)


Tạo API Endpoint

In [ ]:
@app.get("/predict_intent")
def predict_intent(message: str):
    prompt_template = (
        "### Instruction:\nClassify the intent of the following banking customer message. Output ONLY the exact intent label in snake_case format.\n\n"
        "### Input:\n{message}\n\n### Response:\n"
    )
    prompt = prompt_template.format(message=message)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=32, pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    prediction = decoded.split("### Response:")[-1].strip()
    label = prediction.split('\n')[0].strip().lower()
    label = re.sub(r'[^a-z0-9]', '_', label)
    label = re.sub(r'_+', '_', label).strip('_')

    return {"intent": label}

Chạy FastAPI Server ngầm trên cổng 8001

In [ ]:
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="error")

threading.Thread(target=run_api, daemon=True).start()
print("✅ FastAPI Server đã sẵn sàng trên cổng 8001!")

Khởi chạy Pinggy Tunnel

In [ ]:
import pinggy
import time

tunnel = pinggy.start_tunnel(forwardto="localhost:8001")

print(f"Intent API: {tunnel.urls[0]}")

print("Đang giữ kết nối...")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nĐã đóng kết nối Pinggy.")